# Обучение weights.json на Google Colab

**Раннтайм:** Runtime -> Change runtime type -> CPU (GPU/TPU тут не нужны, вся тренировка — это sparse matmul через BLAS на 768 параметрах).

**Идея пайплайна (2 стадии, из-за обрывов сессий Colab):**
1. **Сбор данных** — самая долгая часть (часы на большом PGN). Design-матрица сохраняется на Google Drive, чтобы не парсить PGN заново, если сессия оборвётся.
2. **Обучение** — быстрая часть (минуты), с чекпоинтами на Drive и `--resume`, можно спокойно переигрывать.

**Перед запуском положите на Google Drive** (в одну папку, путь ниже — `PROJECT_DIR`):
- все `.py`-файлы движка: `features.py`, `tuner.py`, `data_compile.py`, `prepare_data_pgn.py`, `train_from_pgn.py`
- сам датасет `lichess-2400-eval.pgn.zst` ([database.chessmont.com](https://database.chessmont.com/))

Рабочие файлы гоняются на локальном диске `/content` (он быстрее, чем Drive, и это важно при чтении гигабайтного PGN), а на Drive лежат только входной датасет и то, что должно пережить обрыв сессии (design-матрица, чекпоинты, итоговые веса).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [3]:
!pip install -q chess zstandard

In [2]:
import os, shutil

# --- поправьте под свою структуру папок на Drive ---
PROJECT_DIR = '/content/drive/MyDrive/chess_engine'   # тут лежат .py-файлы и .pgn.zst
PGN_NAME = 'lichess-2400-eval.pgn.zst'

MATRIX_DIR = f'{PROJECT_DIR}/matrix'        # design-матрица (стадия 1) -> Drive
CKPT_DIR = f'{PROJECT_DIR}/ckpt'            # чекпоинты обучения (стадия 2) -> Drive
WEIGHTS_OUT = f'{PROJECT_DIR}/weights.json' # итоговые веса -> Drive

WORK_DIR = '/content/work'  # локальный быстрый диск для .py и датасета
os.makedirs(WORK_DIR, exist_ok=True)

PY_FILES = ['features.py', 'tuner.py', 'data_compile.py',
            'prepare_data_pgn.py', 'train_from_pgn.py']
for name in PY_FILES:
    shutil.copy(f'{PROJECT_DIR}/{name}', f'{WORK_DIR}/{name}')

local_pgn = f'{WORK_DIR}/{PGN_NAME}'
if not os.path.exists(local_pgn):
    print('Копирую датасет на локальный диск (один раз, может занять пару минут)...')
    shutil.copy(f'{PROJECT_DIR}/{PGN_NAME}', local_pgn)

os.chdir(WORK_DIR)
print('Готово, рабочая директория:', WORK_DIR)

Готово, рабочая директория: /content/work


## Стадия 1 — сбор данных (запустить один раз)

Парсит PGN, стратифицированно набирает `--limit` позиций и сохраняет design-матрицу в `MATRIX_DIR` на Drive. Если сессия оборвётся посреди этого шага — придётся перезапустить именно эту ячейку с нуля (промежуточного чекпоинта у сбора нет, только у обучения), поэтому для первого прогона стоит начать с небольшого `--limit`/`--max-games`, чтобы проверить, что всё работает, и только потом ставить боевые цифры.

In [13]:
import os
WORKERS = os.cpu_count()  # на бесплатном Colab обычно 2, на Pro/Pro+ бывает больше
print('CPU:', WORKERS)

!python train_from_pgn.py {local_pgn} \
    --limit 200000 \
    --workers {WORKERS} \
    --save-matrix {MATRIX_DIR} \
    --collect-only

CPU: 2
Партий: 6400 | Позиций: 50454/200000 | 863 поз/сек
Партий: 13800 | Позиций: 101500/200000 | 832 поз/сек
Партий: 24400 | Позиций: 151646/200000 | 713 поз/сек
Партий: 374000 | Позиций: 200000/200000 | 52 поз/сек
Заполненность бакетов:
  opening_close             20000 / 20000
  opening_edge              10000 / 10000
  opening_decisive           6000 / 6000
  middlegame_close          40000 / 40000
  middlegame_edge           30000 / 30000
  middlegame_decisive       20000 / 20000
  endgame_close             30000 / 30000
  endgame_edge              24000 / 24000
  endgame_decisive          20000 / 20000
[2026-09-12 23:01:05] Собрано 200000 позиций за 3868.1s
[2026-09-12 23:01:19] Design-матрица: 196000 train / 4000 val, nnz=6248344
[2026-09-12 23:01:24] Design-матрица сохранена в /content/drive/MyDrive/chess_engine/matrix


### Быстрая проверка пайплайна перед боевым прогоном (опционально)
Маленький `--limit`/`--max-games` — если отработает за секунды без ошибок, можно спокойно запускать полную стадию 1 выше.

In [14]:
!python train_from_pgn.py {local_pgn} \
    --limit 2000 --max-games 50 \
    --save-matrix /content/work/matrix_smoketest \
    --collect-only

Заполненность бакетов:
  opening_close               200 / 200
  opening_edge                 12 / 100
  opening_decisive              4 / 60
  middlegame_close            400 / 400
  middlegame_edge             256 / 300
  middlegame_decisive          85 / 200
  endgame_close               300 / 300
  endgame_edge                240 / 240
  endgame_decisive            200 / 200
[2026-09-12 23:01:30] Собрано 1697 позиций за 4.3s
[2026-09-12 23:01:30] Design-матрица: 1664 train / 33 val, nnz=49512
[2026-09-12 23:01:30] Design-матрица сохранена в /content/work/matrix_smoketest


## Стадия 2 — обучение (можно перезапускать сколько угодно раз)

Грузит уже готовую design-матрицу с Drive (PGN заново не парсится), учит веса с чекпоинтами на Drive. Если сессия Colab оборвётся посреди обучения — просто запустите эту же ячейку ещё раз, `--resume` подхватит последний чекпоинт из `CKPT_DIR`.

In [15]:
!python train_from_pgn.py \
    --load-matrix {MATRIX_DIR} \
    --out {WEIGHTS_OUT} \
    --epochs 30 \
    --init-material \
    --checkpoint-dir {CKPT_DIR} \
    --resume

[2026-09-12 23:01:31] Загружена design-матрица из /content/drive/MyDrive/chess_engine/matrix: 196000 train / 4000 val
[2026-09-12 23:01:31] [resume] найден чекпоинт /content/drive/MyDrive/chess_engine/ckpt/ckpt_epoch_0030.npz, продолжаю с эпохи 31/30
[2026-09-12 23:01:31] [resume] чекпоинт уже на эпохе 30 >= epochs=30, обучать нечего
[2026-09-12 23:01:31] Готово. Веса сохранены в /content/drive/MyDrive/chess_engine/weights.json


## Дообучение позже на новых партиях (опционально)
Например, если чуть погодя добавите ещё один `.pgn.zst`: соберите его в отдельную `MATRIX_DIR`-2, а тут просто стартуйте с уже обученных весов.

In [16]:
!python train_from_pgn.py \
    --load-matrix {MATRIX_DIR} \
    --out {WEIGHTS_OUT} \
    --init-weights {WEIGHTS_OUT} \
    --epochs 5

[2026-09-12 23:01:32] Загружена design-матрица из /content/drive/MyDrive/chess_engine/matrix: 196000 train / 4000 val
[2026-09-12 23:01:32] Стартовые веса загружены из /content/drive/MyDrive/chess_engine/weights.json
[2026-09-12 23:01:32] epoch 1/5  mse=0.044981  val_mse=0.045289  time=0.1s
[2026-09-12 23:01:33] epoch 2/5  mse=0.043743  val_mse=0.044569  time=0.1s
[2026-09-12 23:01:33] epoch 3/5  mse=0.042971  val_mse=0.043926  time=0.1s
[2026-09-12 23:01:33] epoch 4/5  mse=0.042323  val_mse=0.043386  time=0.1s
[2026-09-12 23:01:33] epoch 5/5  mse=0.041742  val_mse=0.042874  time=0.1s
[2026-09-12 23:01:33] Готово. Веса сохранены в /content/drive/MyDrive/chess_engine/weights.json


In [17]:
# Итоговые веса уже лежат на Drive (WEIGHTS_OUT), но если хочется сразу
# скачать файл на локальную машину:
from google.colab import files
files.download(WEIGHTS_OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>